<a href="https://colab.research.google.com/github/MukeshReddy8977/AI-COURSE-PROGRAMS-2026/blob/main/WEATHER_TRAVELAGENT_VR2_0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:

!pip -q install groq requests fastapi uvicorn pyngrok python-multipart

In [4]:

import os
from getpass import getpass

os.environ["GROQ_API_KEY"] = getpass("Enter Groq API key: ")
os.environ["NGROK_AUTHTOKEN"] = getpass("Enter ngrok auth token: ")

print("Keys loaded securely for this Colab session.")


Enter Groq API key: ··········
Enter ngrok auth token: ··········
Keys loaded securely for this Colab session.


In [5]:

import sqlite3
import requests
from pathlib import Path

DB_PATH = "/content/atmos_travel.db"

def init_db():
    with sqlite3.connect(DB_PATH, timeout=10) as conn:
        conn.execute("""
            CREATE TABLE IF NOT EXISTS preferences (
                city TEXT PRIMARY KEY,
                interests TEXT NOT NULL,
                travel_style TEXT NOT NULL
            )
        """)
        conn.commit()

def save_preferences(city, interests, travel_style):
    city_key = city.strip().lower()

    with sqlite3.connect(DB_PATH, timeout=10) as conn:
        conn.execute("""
            INSERT INTO preferences (city, interests, travel_style)
            VALUES (?, ?, ?)
            ON CONFLICT(city) DO UPDATE SET
                interests = excluded.interests,
                travel_style = excluded.travel_style
        """, (city_key, interests, travel_style))
        conn.commit()

    return {"status": "saved", "city": city.strip()}

def get_preferences(city):
    with sqlite3.connect(DB_PATH, timeout=10) as conn:
        row = conn.execute("""
            SELECT interests, travel_style
            FROM preferences
            WHERE city = ?
        """, (city.strip().lower(),)).fetchone()

    if row:
        return {
            "city": city,
            "interests": row[0],
            "travel_style": row[1]
        }

    return {
        "city": city,
        "interests": "general sightseeing, local culture and food",
        "travel_style": "balanced",
        "note": "No saved preferences found; general defaults are being used."
    }

WEATHER_CODES = {
    0: "Clear sky",
    1: "Mainly clear",
    2: "Partly cloudy",
    3: "Overcast",
    45: "Fog",
    48: "Depositing rime fog",
    51: "Light drizzle",
    53: "Moderate drizzle",
    55: "Dense drizzle",
    61: "Slight rain",
    63: "Moderate rain",
    65: "Heavy rain",
    71: "Slight snow",
    73: "Moderate snow",
    75: "Heavy snow",
    80: "Rain showers",
    81: "Moderate rain showers",
    82: "Violent rain showers",
    95: "Thunderstorm",
    96: "Thunderstorm with hail",
    99: "Severe thunderstorm with hail"
}

def get_weather(location):
    try:
        geo = requests.get(
            "https://geocoding-api.open-meteo.com/v1/search",
            params={
                "name": location,
                "count": 1,
                "language": "en",
                "format": "json"
            },
            timeout=15
        )
        geo.raise_for_status()
        places = geo.json().get("results", [])

        if not places:
            return {"error": f"Could not find the city: {location}"}

        place = places[0]

        response = requests.get(
            "https://api.open-meteo.com/v1/forecast",
            params={
                "latitude": place["latitude"],
                "longitude": place["longitude"],
                "current": (
                    "temperature_2m,relative_humidity_2m,"
                    "apparent_temperature,precipitation,"
                    "weather_code,wind_speed_10m"
                ),
                "daily": (
                    "temperature_2m_max,temperature_2m_min,"
                    "precipitation_probability_max,weather_code"
                ),
                "forecast_days": 1,
                "timezone": "auto"
            },
            timeout=20
        )
        response.raise_for_status()
        data = response.json()

        current = data.get("current", {})
        daily = data.get("daily", {})
        code = current.get("weather_code")

        return {
            "city": place["name"],
            "region": place.get("admin1", ""),
            "country": place.get("country", ""),
            "timezone": data.get("timezone"),
            "current": current,
            "condition": WEATHER_CODES.get(code, "Unknown condition"),
            "today": daily,
            "source": "Open-Meteo",
            "units": data.get("current_units", {})
        }

    except requests.RequestException as e:
        return {"error": f"Weather service unavailable: {str(e)}"}

init_db()
print("Weather service and SQLite database are ready!")


Weather service and SQLite database are ready!


In [17]:

import json
from groq import Groq

client = Groq(api_key=os.environ["GROQ_API_KEY"])

MODEL = "openai/gpt-oss-120b"

tool_specs = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Retrieve current weather and today's forecast for a city.",
            "parameters": {
                "type": "object",
                "properties": {
                    "location": {
                        "type": "string",
                        "description": "Destination city name"
                    }
                },
                "required": ["location"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_preferences",
            "description": "Retrieve saved interests and travel style for a destination.",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {
                        "type": "string",
                        "description": "Destination city name"
                    }
                },
                "required": ["city"]
            }
        }
    }
]

available_functions = {
    "get_weather": get_weather,
    "get_preferences": get_preferences
}

SYSTEM_PROMPT = """
You are Atmos AI, a professional weather-aware travel advisor.

Always retrieve weather data and saved preferences for the destination
before creating a personalized itinerary.

Format your answer with these headings:

1. Weather Overview
2. Travel Suitability
3. Suggested Itinerary
4. Weather Precautions
5. Travel Tips
6. Final Recommendation

Use clear language, concise bullet points and a chronological itinerary.

Never invent weather measurements, attraction opening hours, ticket prices,
transport availability or other live information.
Distinguish actual retrieved data from estimates.
If a tool fails, clearly explain what could not be verified.
If preferences are defaults, do not pretend they were saved by the user.
Recommend safe alternatives during severe weather.
"""

def run_agent(question, destination):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {
            "role": "user",
            "content": (
                f"Destination: {destination}\n"
                f"Request: {question}\n"
                "Use tools for current weather and destination preferences."
            )
        }
    ]

    for _ in range(6):
        response = client.chat.completions.create(
            model=MODEL,
            messages=messages,
            tools=tool_specs,
            tool_choice="auto",
            temperature=0.3
        )

        message = response.choices[0].message
        messages.append(message)

        if not message.tool_calls:
            return message.content or "No response was generated."

        for tool_call in message.tool_calls:
            name = tool_call.function.name

            try:
                args = json.loads(tool_call.function.arguments)
                function = available_functions.get(name)

                if function is None:
                    output = {"error": "Unknown tool."}
                else:
                    output = function(**args)

            except Exception as e:
                output = {"error": str(e)}

            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "name": name,
                "content": json.dumps(output, default=str)
            })

    return "The agent reached its tool-call limit. Please try again."

print("Groq AI agent and function calling are ready!")


Groq AI agent and function calling are ready!


In [19]:

from fastapi import FastAPI, Form
from fastapi.responses import HTMLResponse
from html import escape

app = FastAPI(title="Atmos AI Travel Advisor")

def make_page(result_html=""):
    page = r"""
<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>Atmos AI | Intelligent Travel Planner</title>
<style>
@import url('https://fonts.googleapis.com/css2?family=Inter:wght@400;500;600;700;800&display=swap');

:root {
  --bg: #080d19;
  --panel: #101829;
  --border: #25334a;
  --text: #f1f5ff;
  --muted: #9aa9c3;
  --blue: #60a5fa;
  --purple: #a78bfa;
  --green: #6ee7b7;
}

* { box-sizing: border-box; }

body {
  margin: 0;
  min-height: 100vh;
  color: var(--text);
  font-family: Inter, Arial, sans-serif;
  background:
    radial-gradient(ellipse at 12% 0%, #172d53 0%, transparent 38%),
    radial-gradient(ellipse at 95% 30%, #281746 0%, transparent 32%),
    var(--bg);
}

header {
  padding: 20px 6%;
  display: flex;
  justify-content: space-between;
  align-items: center;
  border-bottom: 1px solid #ffffff12;
}

.brand {
  display: flex;
  align-items: center;
  gap: 12px;
  font-weight: 800;
  letter-spacing: -.5px;
  font-size: 20px;
}

.logo {
  width: 43px;
  height: 43px;
  display: grid;
  place-items: center;
  border-radius: 14px;
  font-size: 24px;
  background: linear-gradient(135deg, #3b82f6, #8b5cf6);
}

.status {
  color: var(--green);
  font-size: 11px;
  font-weight: 700;
  letter-spacing: .5px;
  border: 1px solid #6ee7b733;
  background: #6ee7b710;
  padding: 10px 14px;
  border-radius: 30px;
}

main {
  max-width: 1160px;
  margin: auto;
  padding: 52px 24px 65px;
}

.hero {
  text-align: center;
  max-width: 780px;
  margin: 0 auto 42px;
}

.eyebrow {
  display: inline-block;
  color: #bfdbfe;
  background: #3b82f61a;
  border: 1px solid #60a5fa35;
  padding: 9px 15px;
  border-radius: 30px;
  font-size: 11px;
  font-weight: 700;
  letter-spacing: 1px;
}

h1 {
  font-size: clamp(39px, 6vw, 66px);
  line-height: 1.08;
  letter-spacing: -2.8px;
  margin: 22px 0 17px;
}

.gradient {
  background: linear-gradient(90deg, #60a5fa, #c4b5fd, #6ee7b7);
  -webkit-background-clip: text;
  background-clip: text;
  color: transparent;
}

.hero p {
  color: var(--muted);
  font-size: 15px;
  line-height: 1.9;
  max-width: 620px;
  margin: auto;
}

.layout {
  display: grid;
  grid-template-columns: 1.08fr .92fr;
  gap: 22px;
  align-items: stretch;
}

.panel {
  background: linear-gradient(145deg, #131e32, #0d1423);
  border: 1px solid var(--border);
  border-radius: 22px;
  padding: 27px;
  box-shadow: 0 20px 65px #00000020;
}

.panel h2 {
  margin: 0 0 9px;
  font-size: 21px;
}

.sub {
  color: var(--muted);
  font-size: 13px;
  line-height: 1.8;
  margin: 0 0 25px;
}

label {
  display: block;
  margin: 18px 0 9px;
  color: #dbe6f8;
  font-size: 12px;
  font-weight: 700;
  letter-spacing: .5px;
}

input, textarea, select {
  width: 100%;
  padding: 14px;
  color: var(--text);
  background: #080f1e;
  border: 1px solid #2a3b55;
  border-radius: 11px;
  font-family: inherit;
  font-size: 14px;
  outline: none;
}

input:focus, textarea:focus, select:focus {
  border-color: var(--blue);
  box-shadow: 0 0 0 3px #60a5fa18;
}

textarea {
  resize: vertical;
  min-height: 105px;
  line-height: 1.7;
}

input::placeholder, textarea::placeholder {
  color: #667792;
}

select option {
  background: #101829;
}

button {
  width: 100%;
  margin-top: 22px;
  padding: 16px;
  color: white;
  font-family: inherit;
  font-size: 14px;
  font-weight: 700;
  border: 0;
  border-radius: 12px;
  cursor: pointer;
  background: linear-gradient(100deg, #2563eb, #7c3aed);
  box-shadow: 0 8px 24px #4f46e533;
  transition: transform .2s, filter .2s;
}

button:hover {
  transform: translateY(-2px);
  filter: brightness(1.12);
}

.side {
  display: flex;
  flex-direction: column;
  gap: 14px;
}

.feature {
  padding: 19px;
  display: flex;
  align-items: flex-start;
  gap: 15px;
  background: #ffffff04;
  border: 1px solid #ffffff10;
  border-radius: 16px;
  transition: border-color .2s, transform .2s;
}

.feature:hover {
  border-color: #60a5fa55;
  transform: translateY(-2px);
}

.icon {
  width: 45px;
  height: 45px;
  flex-shrink: 0;
  display: grid;
  place-items: center;
  border-radius: 13px;
  background: #60a5fa15;
  font-size: 23px;
}

.feature h3 {
  font-size: 14px;
  margin: 2px 0 7px;
}

.feature p {
  color: var(--muted);
  font-size: 12px;
  line-height: 1.8;
  margin: 0;
}

.results {
  margin-top: 25px;
}

.weather-heading {
  font-size: 20px;
  margin: 0 0 14px;
}

.weather-grid {
  display: grid;
  grid-template-columns: repeat(3, minmax(0, 1fr));
  gap: 12px;
  margin-bottom: 20px;
}

.metric {
  min-width: 0;
  padding: 17px;
  border: 1px solid #293954;
  border-radius: 15px;
  background: linear-gradient(145deg, #13213a, #0c1423);
}

.metric .label {
  color: var(--muted);
  font-size: 11px;
  margin-bottom: 11px;
}

.metric .value {
  font-size: clamp(18px, 2.4vw, 26px);
  font-weight: 800;
  overflow-wrap: anywhere;
}

.metric .unit {
  color: #9ab4d9;
  font-size: 11px;
  margin-top: 7px;
}

.result {
  padding: 25px;
  background: linear-gradient(145deg, #111d32, #0c1423);
  border: 1px solid #3b5275;
  border-radius: 18px;
  overflow-wrap: anywhere;
}

.result h2 {
  margin-top: 0;
  color: #93c5fd;
}

.answer {
  color: #d5deee;
  white-space: pre-wrap;
  overflow-wrap: anywhere;
  font-family: inherit;
  font-size: 14px;
  line-height: 1.95;
  margin: 0;
}

footer {
  padding: 24px;
  border-top: 1px solid #ffffff10;
  text-align: center;
  color: #71819b;
  font-size: 11px;
  line-height: 2;
}

@media(max-width: 780px) {
  header { padding: 16px; }
  main { padding: 38px 15px 45px; }
  .layout { grid-template-columns: 1fr; }
  h1 { letter-spacing: -1.5px; }
  .panel { padding: 21px; }
  .weather-grid { grid-template-columns: repeat(2, minmax(0, 1fr)); }
  .status { font-size: 9px; }
}

@media(max-width: 380px) {
  .weather-grid { grid-template-columns: 1fr; }
}
</style>
</head>

<body>
<header>
  <div class="brand">
    <div class="logo">☁</div>
    <span>ATMOS <span style="color:#93c5fd">AI</span></span>
  </div>
  <div class="status">✦ AI TRAVEL ASSISTANT</div>
</header>

<main>
  <section class="hero">
    <div class="eyebrow">✦ SMARTER TRIPS START HERE</div>
    <h1>Travel with<br><span class="gradient">Intelligence.</span></h1>
    <p>
      Your personal AI travel companion. Discover weather-aware
      itineraries, personalized recommendations and smarter ways
      to explore every destination.
    </p>
  </section>

  <section class="layout">
    <div class="panel">
      <h2>Plan your next adventure ✈️</h2>
      <p class="sub">
        Choose a destination and your travel preferences.
        Atmos AI will combine weather information with your plans.
      </p>

      <form action="/ask" method="post">
        <label for="city">DESTINATION</label>
        <input id="city" name="city"
          placeholder="e.g. Visakhapatnam, Hyderabad"
          required maxlength="100">

        <label for="interests">YOUR INTERESTS (OPTIONAL)</label>
        <input id="interests" name="interests"
          placeholder="Nature, beaches, history, food..."
          maxlength="300">

        <label for="travel_style">TRAVEL STYLE</label>
        <select id="travel_style" name="travel_style">
          <option value="budget-friendly">Budget-friendly</option>
          <option value="balanced" selected>Balanced</option>
          <option value="comfort and relaxation">Comfort & relaxation</option>
          <option value="adventure">Adventure</option>
        </select>

        <label for="question">WHAT DO YOU HAVE IN MIND?</label>
        <textarea id="question" name="question"
          placeholder="Plan a one-day trip with suitable activities and weather precautions..."
          required maxlength="1500"></textarea>

        <button type="submit">Generate My Travel Plan &nbsp; →</button>
      </form>
    </div>

    <div class="side">
      <div class="feature">
        <div class="icon">🌤️</div>
        <div>
          <h3>Weather Intelligence</h3>
          <p>Current weather and forecast information for informed travel decisions.</p>
        </div>
      </div>

      <div class="feature">
        <div class="icon">🧠</div>
        <div>
          <h3>AI-Powered Planning</h3>
          <p>Groq-powered function calling turns retrieved information into useful advice.</p>
        </div>
      </div>

      <div class="feature">
        <div class="icon">🎯</div>
        <div>
          <h3>Personalized Suggestions</h3>
          <p>Save your interests and travel style in the SQLite database.</p>
        </div>
      </div>

      <div class="feature">
        <div class="icon">🛡️</div>
        <div>
          <h3>Weather-Aware Decisions</h3>
          <p>Get practical precautions and indoor or outdoor activity suggestions.</p>
        </div>
      </div>
    </div>
  </section>

  RESULT_PLACEHOLDER
</main>

<footer>
  ATMOS AI · SMART WEATHER &amp; TRAVEL ADVISORY AGENT<br>
  Python · FastAPI · Groq AI · Open-Meteo · SQLite
</footer>
</body>
</html>
"""
    return page.replace("RESULT_PLACEHOLDER", result_html)

@app.get("/", response_class=HTMLResponse)
def home():
    return make_page()

@app.post("/ask", response_class=HTMLResponse)
def ask(
    city: str = Form(...),
    question: str = Form(...),
    interests: str = Form(""),
    travel_style: str = Form("balanced")
):
    city = city.strip()
    question = question.strip()
    interests = interests.strip()

    if not city or not question:
        return HTMLResponse(
            make_page('<div class="result">Enter a destination and a question.</div>'),
            status_code=400
        )

    if interests:
        save_preferences(city, interests, travel_style)

    weather = get_weather(city)

    if "error" in weather:
        weather_html = (
            '<div class="result"><h2>Weather unavailable</h2><p class="answer">'
            + escape(weather["error"]) +
            '</p></div>'
        )
    else:
        current = weather.get("current", {})
        today = weather.get("today", {})

        def first_value(data, key):
            values = data.get(key, [])
            return values[0] if values else None

        temp = current.get("temperature_2m")
        feels = current.get("apparent_temperature")
        humidity = current.get("relative_humidity_2m")
        wind = current.get("wind_speed_10m")
        rain = first_value(today, "precipitation_probability_max")
        high = first_value(today, "temperature_2m_max")
        low = first_value(today, "temperature_2m_min")

        def show(value, suffix=""):
            return "Unavailable" if value is None else f"{value}{suffix}"

        city_name = escape(str(weather.get("city", city)))
        condition = escape(str(weather.get("condition", "Unknown")))

        weather_html = f"""
        <section class="results">
          <h2 class="weather-heading">🌦️ Weather in {city_name}</h2>
          <p class="sub">Live conditions · Source: Open-Meteo</p>
          <div class="weather-grid">
            <div class="metric">
              <div class="label">CURRENT TEMPERATURE</div>
              <div class="value">{show(temp, "°C")}</div>
              <div class="unit">{condition}</div>
            </div>
            <div class="metric">
              <div class="label">FEELS LIKE</div>
              <div class="value">{show(feels, "°C")}</div>
              <div class="unit">Apparent temperature</div>
            </div>
            <div class="metric">
              <div class="label">HUMIDITY</div>
              <div class="value">{show(humidity, "%")}</div>
              <div class="unit">Relative humidity</div>
            </div>
            <div class="metric">
              <div class="label">WIND SPEED</div>
              <div class="value">{show(wind)}</div>
              <div class="unit">km/h</div>
            </div>
            <div class="metric">
              <div class="label">RAIN PROBABILITY</div>
              <div class="value">{show(rain, "%")}</div>
              <div class="unit">Today's maximum forecast</div>
            </div>
            <div class="metric">
              <div class="label">TODAY'S HIGH / LOW</div>
              <div class="value">{show(high, "°")} / {show(low, "°")}</div>
              <div class="unit">Forecast temperatures</div>
            </div>
          </div>
        </section>
        """

    prompt = (
        f"Destination: {city}. User request: {question}. "
        f"Travel style: {travel_style}. "
        "Retrieve current weather and saved preferences using your tools. "
        "Personalize the itinerary and clearly state any limitations."
    )

    try:
        answer = run_agent(prompt, city)
        answer_html = (
            '<section class="result"><h2>✦ Your AI Travel Plan</h2>'
            '<pre class="answer">'
            + escape(str(answer)) +
            '</pre></section>'
        )
    except Exception as e:
        answer_html = (
            '<section class="result"><h2>Travel advisor error</h2>'
            '<pre class="answer">'
            + escape(str(e)) +
            '</pre></section>'
        )

    return make_page(weather_html + answer_html)

print("Atmos AI dark-blue website is ready!")


Atmos AI dark-blue website is ready!


In [21]:

import threading
import time
import requests
import uvicorn
from pyngrok import ngrok

# Start with a clean local ngrok agent
ngrok.kill()
ngrok.set_auth_token(os.environ["NGROK_AUTHTOKEN"])

PORT = 8000

server_config = uvicorn.Config(
    app,
    host="0.0.0.0",
    port=PORT,
    log_level="warning"
)
server = uvicorn.Server(server_config)

server_thread = threading.Thread(
    target=server.run,
    daemon=True
)
server_thread.start()

# Confirm that FastAPI is actually running
for attempt in range(30):
    try:
        response = requests.get(
            f"http://127.0.0.1:{PORT}/",
            timeout=2
        )
        if response.status_code == 200:
            print("FastAPI server is running!")
            break
    except requests.RequestException:
        pass

    time.sleep(1)
else:
    raise RuntimeError(
        "FastAPI did not start. Restart the Colab runtime "
        "and execute Cells 1-5 before trying again."
    )

# Create the public HTTPS tunnel
try:
    tunnel = ngrok.connect(PORT, "http")
    print("\n" + "=" * 55)
    print("YOUR ATMOS AI WEBSITE")
    print(tunnel.public_url)
    print("=" * 55)
    print("Open this URL in a new browser tab.")
except Exception as e:
    print("The FastAPI server is running locally.")
    print("ngrok could not create a public URL.")
    print("Error:", str(e))
    print(
        "If ERR_NGROK_334 appears, stop the existing endpoint "
        "in your ngrok dashboard and rerun this cell."
    )


ERROR:    [Errno 98] error while attempting to bind on address ('0.0.0.0', 8000): address already in use


FastAPI server is running!

YOUR ATMOS AI WEBSITE
https://smoking-kilogram-plop.ngrok-free.dev
Open this URL in a new browser tab.
